# 02 — Trace collection (50-100K (tool, args, output) pairs)

Runs Qwen3-4B in ReAct mode over a 5K-prompt mix and caches every tool call.
All cached pairs end up in `tool_cache.sqlite` and are dumped to a jsonl trace corpus
for predictor training in Notebook 04.

**Time**: ~6-10 hrs on 1× A100. Set `N_PROMPTS` lower for a dry run.


In [ ]:
import sys, os; sys.path.insert(0, str(os.path.abspath(os.path.join(os.getcwd(), '..'))))
import json, time, torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from dyna_grpo.config import MODEL, PATHS, GRPO
from dyna_grpo.data import collection_prompts
from dyna_grpo.trace_collector import rollout
from dyna_grpo.tools import _cache, TOOL_RE, parse_tool_call
from dyna_grpo.utils import write_jsonl, logger

In [ ]:
N_PROMPTS = 5000      # set to 200 for a quick dry run
MAX_TOOL_CALLS = 4

tok = AutoTokenizer.from_pretrained(MODEL.actor_name, trust_remote_code=True)
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(
    MODEL.actor_name, torch_dtype=torch.bfloat16, device_map='cuda:0', trust_remote_code=True).eval()

def gen_fn(ctx, max_new):
    enc = tok(ctx, return_tensors='pt', truncation=True, max_length=6000).to('cuda:0')
    o = model.generate(**enc, max_new_tokens=max_new, do_sample=True,
                       temperature=0.7, top_p=0.9,
                       pad_token_id=tok.pad_token_id)
    return tok.decode(o[0][enc.input_ids.size(1):], skip_special_tokens=True)

In [ ]:
prompts = collection_prompts(n_math=int(N_PROMPTS*0.6),
                              n_code=int(N_PROMPTS*0.3),
                              n_knowledge=int(N_PROMPTS*0.1))
print(f'Total prompts: {len(prompts)}')

t0 = time.time()
n_calls = 0
for i, p in enumerate(tqdm(prompts)):
    try:
        traj = rollout(p['prompt'], gen_fn, max_tool_calls=MAX_TOOL_CALLS,
                       max_new_tokens=2048)
        n_calls += sum(1 for s in traj.segments if s.type == 'tool_call')
    except Exception as e:
        logger.warning(f'rollout {i} failed: {e}')
    if i % 100 == 99:
        print(f'  {i+1} prompts done, {n_calls} tool calls cached, '
              f'elapsed {(time.time()-t0)/60:.1f} min')
print(f'TOTAL: {n_calls} cached tool calls')

In [ ]:
# Dump cache → jsonl trace corpus, split per tool, with train/val/probe splits
import random
from pathlib import Path
random.seed(0)

for tool in ('calc', 'code', 'search'):
    rows = [{'tool': tool, 'args': args, 'output': out}
            for args, out in _cache.all_for(tool)]
    random.shuffle(rows)
    n = len(rows)
    n_val = max(200, n // 20)
    n_probe = max(200, n // 20)
    train, val, probe = rows[: n - n_val - n_probe], rows[n - n_val - n_probe: n - n_probe], rows[n - n_probe:]
    base = Path(PATHS['traces'])
    write_jsonl(base / f'{tool}_train.jsonl', train)
    write_jsonl(base / f'{tool}_val.jsonl', val)
    write_jsonl(base / f'{tool}_probe.jsonl', probe)
    print(f'{tool}: train={len(train)} val={len(val)} probe={len(probe)}')